# **Installs** 

## Libraries and Dependencies

In [63]:
!pip -q install \
  langgraph==0.2.39 \
  langchain==0.3.14 \
  langchain-core==0.3.29 \
  langchain-community==0.3.14 \
  langsmith==0.1.147 \
  openai==1.59.6 \
  langchain-openai==0.2.14 \
  pydantic==2.10.4 \
  pandas==2.2.3 \
  numpy==2.2.1

## Imports

In [64]:
import os              # environment variables / runtime config
import re              # text normalization + lightweight pattern matching
import json            # serialize/parse tool outputs + structured data interchange
import sqlite3         # connect/query the SQLite database
import pandas as pd    # load/query CSV reference data
import numpy as np     # numeric utilities
from typing import Any, Dict, List, Optional, TypedDict, Tuple  # type hints + structured state for agent workflow

from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage  # chat + tool observation messages
from langchain_core.prompts import ChatPromptTemplate  # consistent prompt templates
from langchain_core.tools import tool                 # define LLM-callable tools
from langchain_core.output_parsers import JsonOutputParser  # parse LLM outputs into JSON

from langchain_openai import ChatOpenAI  # OpenAI chat model wrapper

from langgraph.graph import StateGraph, END  # build the LangGraph state machine + termination node

from IPython.display import Image, display  # visualize the graph / notebook-friendly display

# **Tools and functions**

## Generic non-tool utility functions

In [65]:
def readCSV(filePath: str) -> pd.DataFrame:
    """Reads a CSV file and returns its content as a data frame.
        Inputs:
            filePath (str): The file path to the CSV file (e.g., "data/patients.csv").
        Output:
            pandas.DataFrame: A DataFrame containing the data from the CSV file.     
    """
    print(f"Attempting to read CSV file: {filePath}")
    frame = pd.read_csv(filePath) # Read the CSV file into a DataFrame
    # print(frame.head(1))  # Display the first few rows for verification
    print(f"Succesfully read {len(frame)} rows from {filePath}") # Log the number of rows read
    return frame

def readDB(db_path: str) -> sqlite3.Cursor:
    """Connects to the SQLite database and returns a cursor for executing queries.
        Inputs:
            db_path (str): The file path to the SQLite database (e.g., "ehr_database.db").
        Output: 
            sqlite3.Cursor: A cursor object that can be used to execute SQL queries on the database.
    """
    print(f"Connecting to database at: {db_path}")
    conn = sqlite3.connect(db_path)  # Establish a connection to the database
    cursor = conn.cursor()  # Create a cursor object to execute SQL queries
    return cursor

def queryDB(cursor: sqlite3.Cursor, query: str) -> List[Tuple]:
    """Executes a SQL query on the database and returns the results.
        Inputs:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            query (str): The SQL query to be executed (e.g., "SELECT * FROM patients;").
        Output:
            list: A list of tuples containing the results of the query.
    """
    print(f"Executing SQL query: {query}")
    cursor.execute(query)  # Execute the provided SQL query
    results = cursor.fetchall()  # Fetch all results from the executed query
    print(f"Query returned {len(results)} rows")  # Log the number of rows returned
    return results

def queryDBSafe(cursor: sqlite3.Cursor, query: str, params: tuple = ()) -> List[Tuple]:
    """Executes a Parameterized SQL query on the database and returns the results.
        This function is designed to safely execute SQL queries with parameters, preventing SQL injection attacks.
        Inputs:
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
            query (str): The SQL query to be executed, with placeholders for parameters (e.g., "SELECT * FROM patients WHERE age > ?").
            params (tuple): A tuple of parameters to be safely substituted into the query (e.g., (30,)).
        Output:
            list: A list of tuples containing the results of the query.
    """
    cursor.execute(query, params)  # Execute the provided SQL query
    results = cursor.fetchall()  # Fetch all results from the executed query
    print(f"Query returned {len(results)} rows")  # Log the number of rows returned
    return results

def tableCount(tableName: str, cursor: sqlite3.Cursor) -> int:
    """Counts the number of records in a specified table.
        Inputs:
        tableName (str): The name of the database table to count records from.
        cursor (sqlite3.Cursor): An active database cursor for executing queries.
        Output: 
            int: The count of records in the specified table.
    """
    query = f"SELECT COUNT(*) FROM {tableName};"  # SQL query to count records in the table
    print(f"Counting records in table: {tableName}")
    cursor.execute(query)  # Execute the count query
    count = cursor.fetchone()[0]  # Fetch the count result
    print(f"Table {tableName} has {count} records")  # Log the count result
    return count

def normalizeText(text: str) -> str:
    """Normalize text for consistent matching (lowercase + trimmed + whitespace-collapsed).
        Inputs:
        s (str): Any input text (will be cast to string).

    Output:
        str: Normalized text with:
            - leading/trailing whitespace removed
            - converted to lowercase
            - internal / multiple whitespace collapsed to single spaces
    """
    return re.sub(r"\s+", " ", str(text).strip().lower())  # Convert to lowercase and remove special characters

def toJson(obj: Any) -> str:
    """Convert an object to a JSON string.
        Inputs:
        obj (Any): Any Python object (dict/list/str/etc.). If the object contains
            non-JSON-serializable values (e.g., datetime, numpy types), they are
            converted to strings via default=str.

    Output:
        str: JSON-formatted string (UTF-8 friendly, ensure_ascii=False).
    """
    return json.dumps(obj, ensure_ascii=False, default=str)

## Tools for the MCP Servers

### `getPatientInfo`

In [66]:
@tool("getPatientInfo")
def getPatientInfo(cursor: sqlite3.Cursor, patient_id: str):
    """"
        Retrieves information about a patient from the database based on their unique identifier.
        Args:
            patient_id (str): The unique identifier of the patient whose information is being requested.
            cursor (sqlite3.Cursor): An active database cursor for executing queries.

        Output:
            dict: A dictionary containing the patient's information, including:
                - patient_id
                - first_name
                - last_name
                - birth_year
"""

    rows = queryDBSafe(cursor, 
            """
            SELECT
            patient_id, first_name, last_name, birth_year, sex_at_birth,
            preferred_language, health_literacy_level, timezone, created_at
            FROM patients 
            WHERE patient_id = ?
            """, (patient_id))  # Execute a parameterized query to get patient info
    print(f"getPatientInfo: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows[0] if rows else {"error": f"Patient {patient_id} not found"})

### `listPatientEncounters`

In [67]:
@tool("listPatientEncounters")
def listPatientEncounters(cursor: sqlite3.Cursor, patient_id: str, limit: int=5):
    """"
        Retrieves a list of encounters for a specific patient from the database.
        Args:
            patient_id (str): The unique identifier of the patient whose encounters are being requested.
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
        
        Output:
            list: A list of dictionaries, each containing details about an encounter, including:
            - encounter_id
            - encounter_date
            - encounter_type
            - reason_for_visit
            - diagnosis_summary
            - provider_specialty
            - followup_instructions
            - care_team_contact
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
               encounter_id, encounter_date, encounter_type, reason_for_visit,
               diagnosis_summary, provider_specialty, followup_instructions, care_team_contact
            FROM encounters 
            WHERE patient_id = ?
            ORDER BY encounter_date DESC
            LIMIT ?
            """, (patient_id, max(1, min(int(limit), 10))))  # Execute a parameterized query to get patient encounters
    print(f"listPatientEncounters: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows)

## `getRecentClinicalNote`

In [68]:
@tool("getRecentClinicalNotes")
def getRecentClinicalNotes(cursor: sqlite3.Cursor, patient_id: str, note_type: str = "visit_note"):
    """"
        Retrieves a list of recent clinical notes for a specific patient from the database.
        Args:
            patient_id (str): The unique identifier of the patient whose clinical notes are being requested.
            cursor (sqlite3.Cursor): An active database cursor for executing queries.
        
        Output:
            list: A list of dictionaries, each containing details about a clinical note, including:
            - note_id
            - encounter_id
            - note_date
            - author
            - note_content
    """
    rows = queryDBSafe(cursor, 
            """
            SELECT 
               note_id, encounter_id, patient_id, note_type, note_text, created_at, author_role
            FROM clinical_notes 
            WHERE patient_id = ? AND note_type = ?
            ORDER BY created_at DESC
            LIMIT 1
            """, (patient_id, note_type))  # Execute a parameterized query to get recent clinical notes
    print(f"getRecentClinicalNotes: Retrieved {len(rows)} rows for patient_id={patient_id}")  # Log the number of rows retrieved
    return toJson(rows[0] if rows else {"error": f"No {note_type} found for patient {patient_id}"})

## Data files 

### Define and load the provided CSV Data

In [69]:
TRUSTED_SOURCES_CSV = "datafiles/trusted_sources_catalog.csv"    # path to the trusted_sources_catalog.csv file
LAB_EXPLAIN_CSV =  "datafiles/patient_friendly_lab_explanations.csv"    # path to the patient_friendly_lab_explanations.csv file
MED_EDU_CSV =  "datafiles/medication_education.csv"    # path to the medication_education.csv file
POLICY_CSV =  "datafiles/safety_policy_rules.csv"    # path to the safety_policy_rules.csv file

trustedSources = readCSV(TRUSTED_SOURCES_CSV)
labExplain     = readCSV(LAB_EXPLAIN_CSV)
medEdu         = readCSV(MED_EDU_CSV)
policyRules    = readCSV(POLICY_CSV)


Attempting to read CSV file: datafiles/trusted_sources_catalog.csv
Succesfully read 20 rows from datafiles/trusted_sources_catalog.csv
Attempting to read CSV file: datafiles/patient_friendly_lab_explanations.csv
Succesfully read 30 rows from datafiles/patient_friendly_lab_explanations.csv
Attempting to read CSV file: datafiles/medication_education.csv
Succesfully read 30 rows from datafiles/medication_education.csv
Attempting to read CSV file: datafiles/safety_policy_rules.csv
Succesfully read 10 rows from datafiles/safety_policy_rules.csv


### Define and load the Database

In [70]:
DB_PATH = "datafiles/health_portal.db"    # path to the SQLite database file
cursor = readDB(DB_PATH)
allTables = queryDB(cursor, "SELECT name FROM sqlite_master WHERE type='table';")
for t in allTables:
    tableCount(t[0], cursor)

Connecting to database at: datafiles/health_portal.db
Executing SQL query: SELECT name FROM sqlite_master WHERE type='table';
Query returned 6 rows
Counting records in table: patients
Table patients has 10 records
Counting records in table: encounters
Table encounters has 29 records
Counting records in table: clinical_notes
Table clinical_notes has 43 records
Counting records in table: labs
Table labs has 141 records
Counting records in table: medications
Table medications has 56 records
Counting records in table: allergies
Table allergies has 15 records


### Define and load LLM Configuration and load it

In [71]:
# Load the credentials JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                               # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

modelGenerator = ChatOpenAI(model="gpt-40-mini", temperature=0.2)  # Initialize the OpenAI chat model with specified parameters
modelValidator = ChatOpenAI(model="gpt-40", temperature=0.0)  # Initialize a second OpenAI chat model for validation with zero temperature

# Basic verification for the tools

    Does not work with @tool annotation being present. So the bottom section needs to be commented.

In [72]:
# getPatientInfo(cursor, "P001")
# listPatientEncounters(cursor, "P001", limit=3)
# getRecentClinicalNotes(cursor, "P001", note_type="visit_note")